# LeetCode 2270 — Number of Ways to Split Array bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-split-array/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def waysToSplitArray(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đếm các vị trí chia `i` (`0 ≤ i < n − 1`) sao cho **tổng phần trái ≥ tổng phần phải** — bài tập chuẩn về **tổng tiền tố**.
2. Biến đổi điều kiện `left ≥ total − left` thành `2·left ≥ total` để chỉ cần **một** biến chạy.
3. Hiểu vì sao với **số âm**, các vị trí hợp lệ **không** tạo thành một đoạn liên tiếp ⇒ không thể tìm nhị phân, phải duyệt hết.
4. Cẩn thận với biên: phần phải **không rỗng** (`i < n − 1`), dấu `≥` (không phải `>`).
5. Đối chiếu với vét cạn; hiểu tổng có thể tới `±10¹⁰` (vượt `int32`).

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n`. Chỉ số `i` là một **cách chia hợp lệ** nếu:
- tổng `i + 1` phần tử đầu **≥** tổng `n − i − 1` phần tử cuối;
- bên phải có ít nhất một phần tử: `0 ≤ i < n − 1`.

Trả về **số cách chia hợp lệ**.

**Ví dụ 1:** `nums = [10,4,-8,7]` → `2`: `i = 0`: `10 ≥ 3` ✔; `i = 1`: `14 ≥ −1` ✔; `i = 2`: `6 < 7` ✘.

**Ví dụ 2:** `nums = [2,3,1,0]` → `2`: `i = 0`: `2 < 4` ✘; `i = 1`: `5 ≥ 1` ✔; `i = 2`: `6 ≥ 0` ✔.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, `−10⁵ ≤ nums[i] ≤ 10⁵`.

In [1]:
from typing import List
from itertools import accumulate
from bisect import bisect_left
import random, time

examples = [([10, 4, -8, 7], 2), ([2, 3, 1, 0], 2)]
for nums, exp in examples:
    print(nums, [(sum(nums[:i + 1]), sum(nums[i + 1:])) for i in range(len(nums) - 1)], "-> kỳ vọng", exp)

[10, 4, -8, 7] [(10, 3), (14, -1), (6, 7)] -> kỳ vọng 2
[2, 3, 1, 0] [(2, 4), (5, 1), (6, 0)] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Tổng tiền tố

`pre[i] = nums[0] + … + nums[i − 1]`; tổng `i + 1` phần tử đầu = `pre[i + 1]`, phần còn lại = `total − pre[i + 1]`.

In [2]:
nums = [10, 4, -8, 7]
pre = list(accumulate(nums, initial=0)); total = pre[-1]
print("pre =", pre, "| total =", total)
for i in range(len(nums) - 1):
    print(f"i={i}: trái = {pre[i + 1]}, phải = {total - pre[i + 1]}")

pre = [0, 10, 14, 6, 13] | total = 13
i=0: trái = 10, phải = 3
i=1: trái = 14, phải = -1
i=2: trái = 6, phải = 7


### 3.2 Biến đổi bất đẳng thức để bớt một biến

`left ≥ right` và `right = total − left` ⇒ `left ≥ total − left` ⇔ `2·left ≥ total`. Chỉ cần theo dõi `left`.

In [3]:
for left, total in [(10, 13), (6, 13), (7, 14)]:
    print(f"left={left}, total={total}: left >= total-left ? {left >= total - left} | 2*left >= total ? {2 * left >= total}")

left=10, total=13: left >= total-left ? True | 2*left >= total ? True
left=6, total=13: left >= total-left ? False | 2*left >= total ? False
left=7, total=14: left >= total-left ? True | 2*left >= total ? True


### 3.3 Đếm bằng `sum` trên biểu thức boolean

`sum(cond for ...)` đếm số lần điều kiện đúng (vì `True == 1`).

In [4]:
print(sum(x > 0 for x in [3, -1, 2, 0, 5]))

3


### 3.4 Giá trị lớn

`|tổng| ≤ 10⁵ · 10⁵ = 10¹⁰` > `2³¹ − 1` ⇒ C++/Java phải dùng `long`. Python không tràn.

In [5]:
print(10**5 * 10**5, ">", 2**31 - 1)

10000000000 > 2147483647


### 3.5 `nums[:-1]` tạo bản sao — `range(n − 1)` thì không

Cắt lát tạo **list mới** (O(n) bộ nhớ và thời gian sao chép). Khi chỉ cần duyệt, dùng chỉ số hoặc `itertools.islice` để không sao chép.

In [6]:
from itertools import islice
import sys as _sys
a = list(range(10**5))
print("kích thước bản sao a[:-1]:", _sys.getsizeof(a[:-1]), "byte | islice không sao chép:", sum(islice(a, len(a) - 1)) == sum(a[:-1]))

kích thước bản sao a[:-1]: 800048 byte | islice không sao chép: True


## 4. Ý tưởng nền tảng / trực giác

Hình dung một vách ngăn chạy từ trái sang phải; tại vị trí chia `i`, phần trái là `nums[0..i]`. Khi vách nhảy một bước, `left += nums[i]`.

Tính tay ví dụ 1 (`total = 13`):

| i | left | right = 13 − left | 2·left ≥ 13? |
|---|---|---|---|
| 0 | 10 | 3 | 20 ≥ 13 ✔ |
| 1 | 14 | −1 | 28 ≥ 13 ✔ |
| 2 | 6 | 7 | 12 ≥ 13 ✘ |

Đáp án 2. Không xét `i = 3` vì phần phải sẽ rỗng.

**Quan sát với số âm:** `left` không đơn điệu (14 → 6 vì gặp −8) ⇒ tập vị trí hợp lệ có thể "lủng lỗ" (ví dụ ✔, ✘, ✔). Với mảng **không âm**, `left` tăng dần ⇒ các vị trí hợp lệ là một **hậu tố** (đoạn cuối) ⇒ có thể tìm nhị phân — đây là trường hợp đặc biệt (Cách 5).

In [7]:
nums = [5, -10, 8, -1, 3, -4, 6]
total = sum(nums); left = 0; marks = []
for i in range(len(nums) - 1):
    left += nums[i]; marks.append("✔" if 2 * left >= total else "✘")
print(nums, "total =", total, "->", " ".join(marks), "(không liên tiếp)")

[5, -10, 8, -1, 3, -4, 6] total = 7 -> ✔ ✘ ✘ ✘ ✔ ✘ (không liên tiếp)


### 4.1 Vẽ "độ chênh" `2·left − total` theo vị trí chia

Vị trí hợp lệ ⇔ độ chênh ≥ 0. Vẽ cho một mảng có số âm để thấy đồ thị lên xuống — các vị trí hợp lệ nằm rải rác:

In [8]:
demo = [4, -6, 5, -2, 7, -9, 3, 1, -4, 2]
total = sum(demo); left = 0; diffs = []
for i in range(len(demo) - 1):
    left += demo[i]; diffs.append(2 * left - total)
print("độ chênh:", diffs)
for level in range(max(diffs), min(diffs) - 1, -2):
    print(f"{level:>4} | " + " ".join("#" if d >= level and level >= 0 else ("o" if d <= level < 0 else " ") for d in diffs))
print("hợp lệ   : " + " ".join("✔" if d >= 0 else "✘" for d in diffs))

độ chênh: [7, -5, 5, 1, 15, -3, 3, 5, -3]
  15 |         #        
  13 |         #        
  11 |         #        
   9 |         #        
   7 | #       #        
   5 | #   #   #     #  
   3 | #   #   #   # #  
   1 | #   # # #   # #  
  -1 |   o       o     o
  -3 |   o       o     o
  -5 |   o              
hợp lệ   : ✔ ✘ ✔ ✔ ✔ ✘ ✔ ✔ ✘


Đồ thị trên mức 0 (`#`) là các vị trí hợp lệ, dưới mức 0 (`o`) là không hợp lệ. Vì đường gấp khúc dao động, không có "điểm cắt" duy nhất chia hai vùng ⇒ phải duyệt hết.

## 5. Cách 1 — Brute Force / baseline: tính lại hai tổng cho mỗi vị trí chia

In [9]:
def brute(nums):
    n = len(nums)
    return sum(1 for i in range(n - 1) if sum(nums[:i + 1]) >= sum(nums[i + 1:]))

for nums, exp in examples:
    got = brute(nums)
    print(got, "OK" if got == exp else "SAI")

2 OK
2 OK


**Độ phức tạp:** O(n²) — 10¹⁰ với n = 10⁵ ⇒ TLE.

**Nút thắt:** tổng `nums[:i+1]` và `nums[:i+2]` chỉ khác một phần tử. Cộng dồn ⇒ O(1) mỗi bước.

In [10]:
for n in [2000, 6000]:
    a = [random.randint(-10**5, 10**5) for _ in range(n)]
    t = time.perf_counter(); brute(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 26 ms


n=6000: 292 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `left_i = left_{i−1} + nums[i]` và điều kiện `2·left_i ≥ total`. Một lượt, O(1) bộ nhớ.

**Quy tắc:** với `i = 0..n−2`: `left += nums[i]`; **nếu** `2·left ≥ total` **thì** `ans += 1`.

**Bất biến:** sau bước `i`, `left = Σ nums[0..i]` và `ans` = số vị trí hợp lệ trong `0..i`.

**Ví dụ minh họa vì sao "tổng" chứ không phải "phần tử cuối" quyết định:** ở `[10, 4, −8, 7]`, vị trí `i = 1` hợp lệ vì `10 + 4 = 14 ≥ −8 + 7 = −1`, dù phần tử ngay sau vách (`−8`) chẳng liên quan gì tới quyết định. Mọi thông tin cần thiết được nén vào **một** con số `left` (vì `right` suy ra từ `total`).

**Mối liên hệ với "điểm cân bằng" (pivot index, LC 724):** pivot là vị trí mà tổng trái = tổng phải (không tính phần tử giữa). Bài này nới thành bất đẳng thức và tính phần tử `i` vào bên trái — cùng một kỹ thuật một biến chạy.

**Tại sao dừng ở `n − 2`?** Vị trí `n − 1` cho phần phải rỗng — đề cấm. (Nếu lỡ xét, `2·total ≥ total` ⇔ `total ≥ 0` có thể đúng ⇒ đếm thừa — Lỗi 1.)

**Trường hợp đặc biệt (không âm):** `left` không giảm ⇒ điều kiện `2·left ≥ total` đơn điệu ⇒ tìm nhị phân trên mảng tiền tố cho vị trí đầu tiên thỏa; đáp án = `(n − 1) − vị_trí_đó`. Với số âm thì **không** dùng được.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng tiền tố

In [11]:
def prefix_array(nums):
    pre = list(accumulate(nums, initial=0)); total = pre[-1]; n = len(nums)
    return sum(1 for i in range(n - 1) if pre[i + 1] >= total - pre[i + 1])

for nums, exp in examples:
    got = prefix_array(nums)
    print(got, "OK" if got == exp else "SAI")

2 OK
2 OK


**Độ phức tạp cách 2:** O(n) thời gian, O(n) bộ nhớ (mảng `pre` dài `n + 1`).

### 7.2 Cách 3 — Hai biến chạy `left`, `right` (cách doocs)

In [12]:
def two_running(nums):
    left, right = 0, sum(nums); ans = 0
    for x in nums[:-1]:
        left += x; right -= x
        ans += left >= right
    return ans

for nums, exp in examples:
    got = two_running(nums)
    print(got, "OK" if got == exp else "SAI")

2 OK
2 OK


`nums[:-1]` tạo bản sao (O(n) bộ nhớ). Có thể duyệt `range(n − 1)` để tránh.

### 7.3 Cách 4 — Một biến với `2·left ≥ total`

In [13]:
def one_variable(nums):
    total = sum(nums); left = 0; ans = 0
    for i in range(len(nums) - 1):
        left += nums[i]
        if 2 * left >= total:
            ans += 1
    return ans

for nums, exp in examples:
    got = one_variable(nums)
    print(got, "OK" if got == exp else "SAI")

2 OK
2 OK


**Độ phức tạp cách 4:** O(n) thời gian (hai lượt: một lượt `sum`, một lượt đếm), O(1) bộ nhớ phụ.

### 7.4 Cách 5 — Biểu thức gọn với `accumulate`

In [14]:
def pythonic(nums):
    total = sum(nums)
    return sum(2 * s >= total for s in accumulate(nums[:-1]))

for nums, exp in examples:
    print(pythonic(nums), "OK" if pythonic(nums) == exp else "SAI")

2 OK
2 OK


`accumulate(nums[:-1])` sinh `left_0, left_1, …, left_{n−2}` — đúng các vị trí chia hợp lệ. Vòng lặp nằm trong C ⇒ nhanh.

### 7.5 Cách 6 — Tìm nhị phân (CHỈ khi mọi phần tử không âm)

In [15]:
def binary_search_nonneg(nums):
    assert all(x >= 0 for x in nums)
    pre = list(accumulate(nums))[:-1]          # left_0..left_{n-2}, không giảm
    total = sum(nums)
    # vị trí đầu tiên có 2*left >= total  <=> left >= ceil(total / 2)
    need = (total + 1) // 2
    first = bisect_left(pre, need)
    return len(pre) - first

for nums in [[2, 3, 1, 0], [1, 1, 1, 1], [0, 0], [5, 0, 0, 0]]:
    print(nums, binary_search_nonneg(nums), one_variable(nums))

[2, 3, 1, 0] 2 2
[1, 1, 1, 1] 2 2
[0, 0] 1 1
[5, 0, 0, 0] 3 3


`2·left ≥ total` ⇔ `left ≥ ⌈total/2⌉` (với số nguyên). `bisect_left` tìm vị trí đầu tiên `≥ need`; mọi vị trí sau đó cũng thỏa (đơn điệu). Mục 13 cho thấy áp dụng cách này cho mảng có số âm sẽ sai.

### 7.6 Cách 7 — Hai con trỏ / quét ngược cho mảng không âm (không cần `bisect`)

Với mảng không âm, các vị trí hợp lệ là một hậu tố. Quét từ **phải sang trái**, giữ `right`; vị trí `i` hợp lệ ⇔ `total − right_i ≥ right_i`. Lần đầu (từ phải) gặp vị trí **không** hợp lệ thì mọi vị trí bên trái cũng không hợp lệ ⇒ dừng sớm.

In [16]:
def scan_from_right_nonneg(nums):
    assert all(x >= 0 for x in nums)
    total = sum(nums); right = 0; ans = 0
    for i in range(len(nums) - 2, -1, -1):          # vị trí chia i, phần phải nums[i+1..]
        right += nums[i + 1]
        if total - right >= right:
            ans += 1
        else:
            break                                   # đơn điệu: bên trái càng nhỏ
    return ans

for nums in [[2, 3, 1, 0], [1, 1, 1, 1], [0, 0], [5, 0, 0, 0], [1, 2, 3, 4]]:
    print(nums, scan_from_right_nonneg(nums), Solution().waysToSplitArray(nums) if 'Solution' in globals() else one_variable(nums))

[2, 3, 1, 0] 2 2
[1, 1, 1, 1] 2 2
[0, 0] 1 1
[5, 0, 0, 0] 3 3
[1, 2, 3, 4] 1 1


Dừng sớm giúp nhanh khi đáp án nhỏ; nhưng chỉ đúng khi mọi phần tử **không âm** — với số âm, quét tiếp có thể gặp lại vị trí hợp lệ.

## 8. TRACE từng bước

Trace cách 4 trên `nums = [5, -10, 8, -1, 3, -4, 6]` (`total = 7`).

In [17]:
def trace(nums):
    total = sum(nums); left = 0; ans = 0
    print(f"total = {total}")
    for i in range(len(nums) - 1):
        left += nums[i]; ok = 2 * left >= total; ans += ok
        print(f"i={i}: +{nums[i]:>3} -> left={left:>3}, right={total - left:>3}, 2*left={2*left:>3} >= {total}? {'✔' if ok else '✘'}  (ans={ans})")
    return ans

print("Kết quả:", trace([5, -10, 8, -1, 3, -4, 6]))

total = 7
i=0: +  5 -> left=  5, right=  2, 2*left= 10 >= 7? ✔  (ans=1)
i=1: +-10 -> left= -5, right= 12, 2*left=-10 >= 7? ✘  (ans=1)
i=2: +  8 -> left=  3, right=  4, 2*left=  6 >= 7? ✘  (ans=1)
i=3: + -1 -> left=  2, right=  5, 2*left=  4 >= 7? ✘  (ans=1)
i=4: +  3 -> left=  5, right=  2, 2*left= 10 >= 7? ✔  (ans=2)
i=5: + -4 -> left=  1, right=  6, 2*left=  2 >= 7? ✘  (ans=2)
Kết quả: 2


Trace ví dụ 2 (không âm — các ✔ nằm liền ở cuối):

In [18]:
print("Kết quả:", trace([2, 3, 1, 0]))

total = 6
i=0: +  2 -> left=  2, right=  4, 2*left=  4 >= 6? ✘  (ans=0)
i=1: +  3 -> left=  5, right=  1, 2*left= 10 >= 6? ✔  (ans=1)
i=2: +  1 -> left=  6, right=  0, 2*left= 12 >= 6? ✔  (ans=2)
Kết quả: 2


## 9. Code nộp LeetCode

In [19]:
from typing import List

class Solution:
    def waysToSplitArray(self, nums: List[int]) -> int:
        total = sum(nums)
        left = 0
        ans = 0
        for i in range(len(nums) - 1):        # phần phải phải không rỗng
            left += nums[i]
            if 2 * left >= total:              # left >= total - left
                ans += 1
        return ans

In [20]:
sol = Solution()
for nums, exp in examples:
    got = sol.waysToSplitArray(nums)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
2 | kỳ vọng 2 | OK


## 10. Vì sao thuật toán đúng

**Bất biến.** Ngay sau `left += nums[i]` ở vòng `i`, `left = Σ_{t ≤ i} nums[t]` (quy nạp: đúng với `i = 0`; mỗi vòng thêm đúng `nums[i]`). Khi đó tổng phần phải là `total − left`, và điều kiện đề `left ≥ total − left` tương đương `2·left ≥ total` (cộng `left` hai vế — phép biến đổi tương đương với số nguyên, không chia nên không mất chính xác).

**Phạm vi.** Vòng lặp duyệt đúng `i = 0..n−2` — mọi vị trí chia có phần phải không rỗng, mỗi vị trí đúng một lần ⇒ `ans` đếm đúng. ∎

**Một cách nhìn khác — đếm phần bù.** Số vị trí **không** hợp lệ là số `i < n − 1` có `2·left_i < total`. Tổng hai số này là `n − 1`. Đôi khi đếm phần bù dễ hơn (ví dụ biến thể "trái < phải" chỉ cần lấy `(n − 1) − đáp_án`). Mục 12.3 dùng quan hệ này giữa mảng và mảng đảo dấu.

**Vì sao không cần xét thứ tự hay cấu trúc gì thêm?** Mỗi vị trí chia được quyết định độc lập bởi đúng một con số `left_i` — không có tương tác giữa các vị trí. Do đó bài toán thuần túy là "tính `n − 1` tổng tiền tố và đếm", và O(n) là tối ưu (phải đọc mọi phần tử ít nhất một lần, vì thay đổi bất kỳ phần tử nào cũng có thể đổi đáp án).

**Cách 6 (không âm).** Với `nums[i] ≥ 0`, `left_i` không giảm ⇒ tập `{i : left_i ≥ need}` là một hậu tố của `0..n−2` ⇒ số phần tử = `(n − 1) − (chỉ số đầu tiên)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | O(n²) | O(n) lát cắt |
| 2. Mảng tiền tố | O(n) | O(n) |
| 3. Hai biến | O(n) | O(n) nếu dùng `nums[:-1]` / O(1) nếu dùng chỉ số |
| 4. Một biến | O(n) | O(1) |
| 5. accumulate | O(n) | O(n) (bản sao) |
| 6. Nhị phân (không âm) | O(n) dựng + O(log n) | O(n) |

In [21]:
rng = random.Random(2270)
big = [rng.randint(-10**5, 10**5) for _ in range(10**5)]
for name, f in [("prefix_array", prefix_array), ("two_running", two_running), ("one_variable", one_variable),
                ("pythonic", pythonic), ("Solution", Solution().waysToSplitArray)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<13} -> {r}  {dt*1000:.1f} ms")

prefix_array  -> 18585  14.2 ms
two_running   -> 18585  10.8 ms
one_variable  -> 18585  8.8 ms
pythonic      -> 18585  10.0 ms
Solution      -> 18585  8.9 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
ALL = {"brute": brute, "prefix_array": prefix_array, "two_running": two_running,
       "one_variable": one_variable, "pythonic": pythonic, "Solution": Solution().waysToSplitArray}
edge = examples + [
    ([1, 1], 1),                      # 1 >= 1
    ([1, 2], 0),
    ([2, 1], 1),
    ([0, 0, 0], 2),                   # 0 >= 0 ở mọi vị trí
    ([-1, -1], 1),                    # -1 >= -1
    ([-5, 10], 0),
    ([10**5] * 4, 2),                 # i=1: 2e5 >= 2e5, i=2: 3e5 >= 1e5
    ([-10**5] * 4, 2),                # i=0: -1e5 >= -3e5 ✔, i=1 ✔(=), i=2 ✘ -> tính tay
    ([5, -10, 8, -1, 3, -4, 6], None),
]
for nums, exp in edge:
    if exp is None: exp = brute(nums)
    for name, f in ALL.items():
        a = list(nums)
        assert f(a) == exp, (name, nums, f(list(nums)), exp)
        assert a == nums
print("Edge case OK:", [Solution().waysToSplitArray(list(x)) for x, _ in edge])

Edge case OK: [2, 2, 1, 0, 1, 2, 1, 0, 2, 2, 2]


Kiểm tay `[-10⁵] * 4` (`total = −4·10⁵`): `i = 0`: trái `−10⁵`, phải `−3·10⁵` ⇒ ✔; `i = 1`: `−2·10⁵ ≥ −2·10⁵` ✔; `i = 2`: `−3·10⁵ ≥ −10⁵` ✘ ⇒ **2**.

### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [23]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(2, 30); hi = rng.choice([1, 5, 100, 10**5])
    nums = [rng.randint(-hi, hi) for _ in range(n)]
    ref = brute(nums)
    for name, f in ALL.items():
        assert f(list(nums)) == ref, (name, nums)
    if all(x >= 0 for x in nums):
        assert binary_search_nonneg(nums) == ref
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Tính chất: đảo dấu mọi phần tử ⇒ đếm "trái ≤ phải"; tổng hai đáp án = `(n − 1) + #{left = right}`

In [24]:
rng = random.Random(7)
for _ in range(500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(2, 25))]
    a = Solution().waysToSplitArray(nums); b = Solution().waysToSplitArray([-x for x in nums])
    total = sum(nums); eq = sum(1 for s in accumulate(nums[:-1]) if 2 * s == total)
    assert a + b == len(nums) - 1 + eq
print("tính chất đối xứng OK")

tính chất đối xứng OK


Giải thích: với mảng đảo dấu, điều kiện thành `−left ≥ −right` ⇔ `left ≤ right`. Mỗi vị trí thỏa ít nhất một trong hai điều kiện, và thỏa cả hai ⇔ `left = right`.

### 12.4 Cực trị theo ràng buộc

- `n = 10⁵`, mọi phần tử `10⁵`: vị trí `i` hợp lệ ⇔ `i + 1 ≥ n − i − 1` ⇔ `i ≥ n/2 − 1` ⇒ đáp án `n/2`.
- Mọi phần tử `−10⁵`: ngược lại ⇔ `i + 1 ≤ n − i − 1` ⇒ đáp án `n/2`.
- `n = 2`: chỉ một vị trí chia.

In [25]:
n = 10**5
for name, f in ALL.items():
    if name == "brute": continue
    assert f([10**5] * n) == n // 2
    assert f([-10**5] * n) == n // 2
    assert f([7, 7]) == 1 and f([-7, 7]) == 0
print("cực trị OK")

cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xét cả vị trí `n − 1` (phần phải rỗng)

In [26]:
def wrong_include_last(nums):
    total = sum(nums); left = 0; ans = 0
    for x in nums:                                       # SAI: gồm i = n-1
        left += x; ans += 2 * left >= total
    return ans

print("bản sai :", wrong_include_last([10, 4, -8, 7]), "| bản đúng:", Solution().waysToSplitArray([10, 4, -8, 7]))

bản sai : 3 | bản đúng: 2


**Sửa:** `range(n − 1)` — bên phải phải có ít nhất một phần tử.

### Lỗi 2 — Dùng `>` thay vì `>=`

In [27]:
def wrong_strict(nums):
    total = sum(nums); left = 0; ans = 0
    for i in range(len(nums) - 1):
        left += nums[i]; ans += 2 * left > total          # SAI
    return ans

print("bản sai :", wrong_strict([1, 1]), "| bản đúng:", Solution().waysToSplitArray([1, 1]))

bản sai : 0 | bản đúng: 1


**Sửa:** đề yêu cầu "greater than **or equal**".

### Lỗi 3 — Tìm nhị phân trên mảng có số âm

In [28]:
def wrong_bisect(nums):
    pre = list(accumulate(nums))[:-1]; total = sum(nums)
    return len(pre) - bisect_left(pre, (total + 1) // 2)  # SAI: pre không đơn điệu

nums = [5, -10, 8, -1, 3, -4, 6]
print("bản sai :", wrong_bisect(nums), "| bản đúng:", Solution().waysToSplitArray(nums))

bản sai : 0 | bản đúng: 2


**Vì sao sai:** `bisect` giả định dãy **đã sắp xếp**; tiền tố với số âm không đơn điệu ⇒ kết quả vô nghĩa. **Sửa:** duyệt tuyến tính.

### Lỗi 4 — So sánh với `total // 2` (chia lấy nguyên làm tròn xuống)

In [29]:
def wrong_floor_half(nums):
    total = sum(nums); left = 0; ans = 0
    for i in range(len(nums) - 1):
        left += nums[i]; ans += left >= total // 2       # SAI: 7 // 2 = 3, nhưng cần left >= 3.5
    return ans

print("bản sai :", wrong_floor_half([3, 4]), "| bản đúng:", Solution().waysToSplitArray([3, 4]))

bản sai : 1 | bản đúng: 0


**Vì sao sai:** với `[3, 4]`, `total = 7`: trái `3`, phải `4` ⇒ không hợp lệ. Điều kiện đúng `left ≥ 7/2 = 3,5` ⇔ `left ≥ 4`; nhưng `7 // 2 = 3` nên bản sai chấp nhận `left = 3`. (Dùng `left >= total / 2` với float thì đúng, nhưng phép nhân `2·left` gọn và an toàn hơn.) **Sửa:** `2·left ≥ total` (không chia), hoặc `left ≥ (total + 1) // 2` (làm tròn lên — cẩn thận với số âm: `//` của Python làm tròn xuống nên `(total + 1)//2` vẫn là trần đúng).

### Lỗi 5 — Chỉ so `nums[i]` với `nums[i + 1]` (nhầm "tổng" thành "phần tử")

In [30]:
def wrong_elements(nums):
    return sum(1 for i in range(len(nums) - 1) if nums[i] >= nums[i + 1])     # SAI

print("ví dụ 2      - bản sai :", wrong_elements([2, 3, 1, 0]), "| bản đúng:", Solution().waysToSplitArray([2, 3, 1, 0]), "(trùng hợp)")
print("[1, 2, 0, 5] - bản sai :", wrong_elements([1, 2, 0, 5]), "| bản đúng:", Solution().waysToSplitArray([1, 2, 0, 5]))

ví dụ 2      - bản sai : 2 | bản đúng: 2 (trùng hợp)
[1, 2, 0, 5] - bản sai : 1 | bản đúng: 0


Hai ví dụ của đề đều cho kết quả trùng hợp (2 = 2) — một lời nhắc nữa rằng qua ví dụ mẫu chưa chắc đã đúng. Với `[1, 2, 0, 5]`: tại `i = 1`, phần tử `2 ≥ 0` nhưng tổng `3 < 5`.

**Sửa:** so **tổng** hai phần.

### Lỗi 6 — Tràn số khi dịch sang ngôn ngữ dùng `int` 32-bit

Python không tràn, nhưng nếu viết C++/Java với `int`, `2·left` có thể vượt `2³¹ − 1` (left tới 10¹⁰). Ta mô phỏng phép tính 32-bit có dấu để thấy kết quả sai:

In [31]:
def to_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x

def ways_int32(nums):
    total = 0
    for x in nums: total = to_int32(total + x)
    left = ans = 0
    for i in range(len(nums) - 1):
        left = to_int32(left + nums[i])
        ans += to_int32(2 * left) >= total
    return ans

big = [10**5] * 30000
print("mô phỏng int32:", ways_int32(big), "| đúng (Python):", Solution().waysToSplitArray(big))

mô phỏng int32: 25737 | đúng (Python): 15000


**Sửa:** dùng `long long` / `long` cho tổng (ở Python tự động đúng).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia mảng thành trái/phải, so sánh tổng" ⇒ **một biến chạy** + `total`.
- Biến đổi bất đẳng thức để **tránh chia** (`2·left ≥ total`).
- Có số âm ⇒ tiền tố không đơn điệu ⇒ không tìm nhị phân / hai con trỏ đơn điệu.
- Đọc kỹ điều kiện phần **không rỗng** và dấu `≥`.
- Khi kết quả trên ví dụ mẫu "đúng", hãy thử thêm một phản ví dụ tự dựng — nhiều bản sai (Lỗi 5) vẫn qua cả hai ví dụ của đề.
- Bài cùng họ: 724 Find Pivot Index, 1991, 2256 Minimum Average Difference, 1712 Ways to Split Array Into Three Subarrays (khó hơn: ba phần, không âm ⇒ nhị phân/hai con trỏ), 2483.

## 15. Biến thể của bài

### 15.1 Biến thể A — Chia thành **ba** phần không rỗng `trái ≤ giữa ≤ phải` (LC 1712, mọi phần tử không âm)

Với `nums ≥ 0`, cố định điểm cắt thứ nhất `i`, các điểm cắt thứ hai hợp lệ tạo một đoạn `[lo, hi]` tìm bằng `bisect` trên tiền tố. O(n log n).

In [32]:
def ways_to_split_three(nums, MOD=10**9 + 7):
    pre = list(accumulate(nums, initial=0)); n = len(nums); total = pre[-1]; ans = 0
    for i in range(1, n - 1):                    # trái = pre[i]  (nums[0..i-1])
        left = pre[i]
        # giữa = pre[j] - left >= left  -> pre[j] >= 2*left ; phải = total - pre[j] >= pre[j] - left -> pre[j] <= (total + left) / 2
        lo = max(i + 1, bisect_left(pre, 2 * left))
        hi = min(n - 1, bisect_left(pre, (total + left) // 2 + 1) - 1)
        if lo <= hi: ans += hi - lo + 1
    return ans % MOD

def ways_three_brute(nums):
    n = len(nums); c = 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            a, b, d = sum(nums[:i]), sum(nums[i:j]), sum(nums[j:])
            c += a <= b <= d
    return c

print(ways_to_split_three([1, 1, 1]), ways_to_split_three([1, 2, 2, 2, 5, 0]), ways_to_split_three([3, 2, 1]))
rng = random.Random(151)
for _ in range(500):
    a = [rng.randint(0, 6) for _ in range(rng.randint(3, 12))]
    assert ways_to_split_three(a) == ways_three_brute(a)
print("ba phần: khớp brute force")

1 3 0
ba phần: khớp brute force


### 15.2 Biến thể B — Trả về vị trí chia có **chênh lệch |trái − phải| nhỏ nhất** (hòa lấy nhỏ nhất)

In [33]:
def best_balance_split(nums):
    total = sum(nums); left = 0; best = None
    for i in range(len(nums) - 1):
        left += nums[i]; d = abs(2 * left - total)
        if best is None or d < best[0]: best = (d, i)
    return best[1]

print(best_balance_split([10, 4, -8, 7]), best_balance_split([1, 2, 3, 4, 5, 5]))

2 3


### 15.3 Biến thể C — Đếm cách chia với trọng số: `left ≥ k · right`

In [34]:
def ways_weighted(nums, k):
    total = sum(nums); left = 0; ans = 0
    for i in range(len(nums) - 1):
        left += nums[i]; ans += left >= k * (total - left)
    return ans

rng = random.Random(153)
for _ in range(300):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(2, 12))]; k = rng.randint(0, 3)
    assert ways_weighted(a, k) == sum(1 for i in range(len(a) - 1) if sum(a[:i + 1]) >= k * sum(a[i + 1:]))
print(ways_weighted([10, 4, -8, 7], 1), "(k = 1 là bài gốc)", ways_weighted([10, 4, -8, 7], 3))

2 (k = 1 là bài gốc) 2


### 15.4 Biến thể D — So sánh **trung bình** hai phần thay vì tổng

`left/(i+1) ≥ right/(n−i−1)` ⇔ `left·(n−i−1) ≥ right·(i+1)` (nhân chéo — các mẫu dương nên không đổi chiều bất đẳng thức). Tránh float hoàn toàn.

In [35]:
from fractions import Fraction
def ways_by_average(nums):
    n = len(nums); total = sum(nums); left = 0; ans = 0
    for i in range(n - 1):
        left += nums[i]; right = total - left
        ans += left * (n - i - 1) >= right * (i + 1)
    return ans

def ways_by_average_brute(nums):
    n = len(nums)
    return sum(1 for i in range(n - 1) if Fraction(sum(nums[:i + 1]), i + 1) >= Fraction(sum(nums[i + 1:]), n - i - 1))

rng = random.Random(154)
for _ in range(500):
    a = [rng.randint(-10, 10) for _ in range(rng.randint(2, 15))]
    assert ways_by_average(a) == ways_by_average_brute(a)
print(ways_by_average([10, 4, -8, 7]), "- khớp brute force (Fraction)")

2 - khớp brute force (Fraction)


### 15.5 Biến thể E — Nhiều truy vấn trên **mảng con** `nums[l..r]`

Dựng tiền tố một lần; mỗi truy vấn duyệt các vị trí chia trong đoạn: `left = pre[i+1] − pre[l]`, `seg_total = pre[r+1] − pre[l]`. O(độ dài đoạn) mỗi truy vấn.

In [36]:
def ways_queries(nums, queries):
    pre = list(accumulate(nums, initial=0)); out = []
    for l, r in queries:
        seg = pre[r + 1] - pre[l]
        out.append(sum(1 for i in range(l, r) if 2 * (pre[i + 1] - pre[l]) >= seg))
    return out

rng = random.Random(155)
for _ in range(300):
    a = [rng.randint(-8, 8) for _ in range(rng.randint(2, 20))]
    qs = []
    for _ in range(3):
        l = rng.randrange(len(a) - 1); r = rng.randrange(l + 1, len(a)); qs.append((l, r))
    assert ways_queries(a, qs) == [brute(a[l:r + 1]) for l, r in qs]
print(ways_queries([10, 4, -8, 7], [(0, 3), (1, 3), (0, 1)]), "- khớp brute force")

[2, 1, 1] - khớp brute force


Muốn nhanh hơn (O(log n) mỗi truy vấn) cần cấu trúc phức tạp hơn (ví dụ cây phân đoạn sắp xếp / merge-sort tree đếm số `pre[j] ≥ ngưỡng` trong đoạn) — ngoài phạm vi bài này.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | cộng lại hai lát cắt | O(n²) | O(n) | kiểm chứng |
| 2. Mảng tiền tố | `pre[i+1]` vs `total − pre[i+1]` | O(n) | O(n) | nhiều truy vấn |
| 3. Hai biến | `left`, `right` | O(n) | O(1) | dễ hiểu |
| 4. Một biến | `2·left ≥ total` | O(n) | O(1) | **mặc định**, không chia, không sao chép |
| 5. accumulate | một dòng | O(n) | O(n) | Pythonic |
| 6. Nhị phân | chỉ khi không âm | O(n) + O(log n) | O(n) | biến thể đơn điệu |
| 7. Quét phải + dừng sớm | chỉ khi không âm | O(n) worst | O(1) | đáp án nhỏ, mảng không âm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `waysToSplitArray([0, 0])` và `waysToSplitArray([-2, -1, -3])`?

**Bài 2 (tính tay):** Lập bảng `left`, `right` cho `[3, -2, 4, -1, 1]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **danh sách** các vị trí chia hợp lệ.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài cách 2 (mảng tiền tố).

**Bài 5 (mini challenge):** Được **xóa đúng một** phần tử khỏi `nums` (`n ≥ 3`) trước khi chia; viết `bt5_todo(nums)` trả về số cách chia hợp lệ **lớn nhất** có thể đạt (O(n²) là được).

In [37]:
def bt3_todo(nums):
    # TODO: danh sách vị trí chia hợp lệ
    pass

def bt4_todo(nums):
    # TODO: mảng tiền tố
    pass

def bt5_todo(nums):
    # TODO: xóa một phần tử để số cách chia lớn nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `[0, 0]`: `0 ≥ 0`. Mức 2: `[-2, -1, -3]`, total −6: `i=0`: −2 ≥ −4 ✔; `i=1`: −3 ≥ −3 ✔. Mức 3: `1` và `2`.

**Bài 2.** Mức 1: total = 5. Mức 2: left: 3, 1, 5, 4. Mức 3: right: 2, 4, 0, 1 ⇒ ✔ ✘ ✔ ✔ ⇒ 3.

**Bài 3.** Mức 1: như cách 4. Mức 2: `append(i)` thay vì đếm. Mức 3: trả list.

**Bài 4.** Mức 1: `pre = accumulate(nums, initial=0)`. Mức 2: duyệt `i < n − 1`. Mức 3: so `pre[i+1]` với `total − pre[i+1]`.

**Bài 5.** Mức 1: thử xóa từng chỉ số. Mức 2: gọi `Solution` trên mảng còn lại. Mức 3: lấy max — O(n²). (Thử thách thêm: làm O(n) bằng cách nhận xét rằng xóa `nums[k]` làm `total` giảm `nums[k]` và các `left_i` với `i ≥ k` giảm `nums[k]` — có thể đếm bằng tiền tố của các điều kiện "dịch ngưỡng", nhưng không bắt buộc.)

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1`, `2`. **Bài 2:** left `3, 1, 5, 4`; right `2, 4, 0, 1` ⇒ `3`.

In [38]:
assert Solution().waysToSplitArray([0, 0]) == 1
assert Solution().waysToSplitArray([-2, -1, -3]) == 2
assert Solution().waysToSplitArray([3, -2, 4, -1, 1]) == 3

def bt3(nums):
    total = sum(nums); left = 0; out = []
    for i in range(len(nums) - 1):
        left += nums[i]
        if 2 * left >= total: out.append(i)
    return out

def bt4(nums):
    return prefix_array(nums)

def bt5(nums):
    return max(Solution().waysToSplitArray(nums[:i] + nums[i + 1:]) for i in range(len(nums)))

rng = random.Random(1970)
for _ in range(1500):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(3, 15))]
    lst = bt3(nums)
    assert len(lst) == brute(nums) and all(sum(nums[:i + 1]) >= sum(nums[i + 1:]) for i in lst)
    assert bt4(nums) == brute(nums)
    assert bt5(nums) == max(brute(nums[:i] + nums[i + 1:]) for i in range(len(nums)))
print("bài 5 ví dụ:", bt5([10, 4, -8, 7]), "| Đáp án bài tập: tất cả assert OK")

bài 5 ví dụ:

 2 | Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
total = sum(nums); left = ans = 0
for i in range(len(nums) - 1):     # phần phải không rỗng
    left += nums[i]
    if 2 * left >= total:          # left >= total - left, không chia
        ans += 1
return ans
```

**Checklist:**
- [ ] `range(n − 1)`, không xét vị trí cuối.
- [ ] `>=` (bằng nhau vẫn hợp lệ).
- [ ] Không chia lấy nguyên `total // 2`; dùng `2·left ≥ total`.
- [ ] Có số âm ⇒ không tìm nhị phân.
- [ ] Đã thử: hai phần tử, toàn 0, toàn âm, xen kẽ âm dương.
- [ ] (C++/Java) dùng kiểu 64-bit cho `total` và `left`.